<a href="https://colab.research.google.com/github/Lessio2006/UEBA_proj/blob/develop/UEBA_research_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Dataset](https://drive.google.com/file/d/1yAQ_0NQKeaXIbFQaJTBVwXGA_zlCrjKY/view?usp=sharing)

In [2]:
!gdown --id 1yAQ_0NQKeaXIbFQaJTBVwXGA_zlCrjKY -O 'system.csv'

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1yAQ_0NQKeaXIbFQaJTBVwXGA_zlCrjKY
To: /content/system.csv
100% 655k/655k [00:00<00:00, 8.90MB/s]


In [3]:
import pandas as pd
import numpy as np
import torch.nn as nn
import torch
import torch.nn.functional as F
from tqdm import tqdm_notebook

In [4]:
df = pd.read_csv('/content/system.csv', encoding='utf-8')
df.head()

,timestamp,hour_sin,hour_cos,weekday_sin,weekday_cos,cpu_mean,cpu_max,ram_mean,ram_max,swap_mean,...,listening_count_mean,time_wait_count_mean,new_connection_count,closed_connection_count,unique_remote_host_count,new_remote_host_count,unique_remote_port_count,public_connection_count_mean,private_connection_count_mean,outbound_inbound_ratio
0,2026-07-17T03:50:35Z,0.975740,-0.218933,-0.433884,-0.900969,14.800000,32.8,68.033333,69.4,1.500000,...,38.000000,18.333333,33,33,33,0,25,75.333333,28.000000,0.586664
1,2026-07-17T03:50:50Z,0.975500,-0.219999,-0.433884,-0.900969,15.700000,18.1,57.066667,65.8,1.500000,...,34.666667,47.666667,112,48,42,10,25,115.000000,22.666667,0.128680
2,2026-07-17T03:51:05Z,0.975259,-0.221066,-0.433884,-0.900969,5.866667,7.1,52.366667,52.4,1.500000,...,33.000000,81.333333,16,38,43,2,17,154.666667,20.000000,2.222904
3,2026-07-17T03:51:20Z,0.975017,-0.222130,-0.433884,-0.900969,9.266667,14.2,52.966667,53.5,1.500000,...,33.000000,60.666667,106,58,56,16,17,166.333333,16.000000,0.203507
4,2026-07-17T03:51:35Z,0.974774,-0.223195,-0.433884,-0.900969,12.433333,17.9,53.733333,54.0,0.833333,...,33.000000,63.000000,39,24,60,7,14,200.333333,14.000000,0.837821


In [5]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is available! Using GPU.")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

CUDA is available! Using GPU.


In [6]:
df = df.drop(columns=['timestamp'])

features_np = df.values  # законвертили в np array

X_tensor = torch.tensor(features_np, dtype=torch.float32)

print("Shape of the PyTorch tensor:", X_tensor.shape)
print("First 5 rows of the PyTorch tensor:")
print(X_tensor[:5])

Shape of the PyTorch tensor: torch.Size([3197, 32])
First 5 rows of the PyTorch tensor:
tensor([[ 9.7574e-01, -2.1893e-01, -4.3388e-01, -9.0097e-01,  1.4800e+01,
          3.2800e+01,  6.8033e+01,  6.9400e+01,  1.5000e+00,  2.6900e+02,
          9.0000e+00,  7.0042e+05,  5.7303e+06,  2.8000e+01,  4.3900e+02,
          8.2069e+04,  1.3989e+05,  3.2300e+02,  3.6000e+02,  1.4133e+02,
          3.3667e+01,  7.8667e+01,  3.8000e+01,  1.8333e+01,  3.3000e+01,
          3.3000e+01,  3.3000e+01,  0.0000e+00,  2.5000e+01,  7.5333e+01,
          2.8000e+01,  5.8666e-01],
        [ 9.7550e-01, -2.2000e-01, -4.3388e-01, -9.0097e-01,  1.5700e+01,
          1.8100e+01,  5.7067e+01,  6.5800e+01,  1.5000e+00,  2.6200e+02,
          5.0000e+00,  1.5714e+07,  2.4285e+07,  3.8300e+02,  1.0140e+03,
          7.1282e+05,  5.5395e+06,  2.9180e+03,  4.2260e+03,  1.7233e+02,
          3.8667e+01,  8.6333e+01,  3.4667e+01,  4.7667e+01,  1.1200e+02,
          4.8000e+01,  4.2000e+01,  1.0000e+01,  2.5000e+01,  

Reparameterezation trick нужен для того, чтобы мы могли считать гралиенты при обучении, понятно что есои в нашей цепочки будет слчайное сэмплирование из PDF, то backprop мы не сомжем использовать.


---
Формула reparameterezation trick:
*  $z = \mu + \sigma \odot \epsilon, \quad \text{где } \epsilon \sim \mathcal{N}(0, 1)$

Формула перехода от $log(Var)$ к $\sigma$:
*  $\sigma = \sqrt{\sigma^2} = \sqrt{\exp(\log(\sigma^2))} = \exp\left(\frac{1}{2} \log(\sigma^2)\right)
$



In [ ]:
# input vector -> mlp -> mean, std -> reparam trick -> output clf
class VariationalAE(nn.Module):
  def __init__(self, input_dim=32, hidden_dim=64, latent_dim=8):
    super().__init__()

    self.encoder = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim // 2),
        nn.ReLU(),
        nn.Linear(hidden_dim // 2, hidden_dim // 4),
        nn.ReLU()
    )

    self.fc_mu = nn.Linear(hidden_dim // 4, latent_dim)
    self.fc_log_var = nn.Linear(hidden_dim // 4, latent_dim)

    self.decoder = nn.Sequential(
        nn.Linear(latent_dim, hidden_dim), # First hidden layer of decoder
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim), # Second hidden layer of decoder
        nn.ReLU(),
        nn.Linear(hidden_dim, input_dim),  # Output layer, reconstructing input_dim features
        # nn.Sigmoid() # Uncomment if input features are normalized to [0, 1] and you want values in [0,1]
    )

    self.input_dim = input_dim
    self.hidden_dim = hidden_dim
    self.latent_dim = latent_dim


  def reparameterize(self, mu, log_var):
    """
    :param mu: mean from the encoder's latent space
    :param log_var: log variance from the encoder's latent space
    :return: A sample from the latent distribution
    """
    std = torch.exp(0.5 * log_var)
    eps = torch.randn_like(std)
    sample = mu + (eps * std)
    return sample


  def encode(self, x):
    h = self.encoder(x) # Pass input through the sequential encoder layers
    mu = self.fc_mu(h)
    log_var = self.fc_log_var(h)
    return mu, log_var

  def decode(self, z): # Decodes a latent space sample z back into original feature space
    reconstruction = self.decoder(z) # Pass latent sample through the sequential decoder layers
    return reconstruction

  def forward(self, x):
    mu, log_var = self.encode(x)
    z = self.reparameterize(mu, log_var)
    reconstruction = self.decode(z)    # Decode the latent sample to reconstruct the input
    return reconstruction, mu, log_var
